In [34]:
!ls /home/gustavo/repos/Schools-2025-February-Peru/


debriefing.md	instrucciones.md   material  pitch-template.md
groupmaking.md	instrucciones.md~  media     README.md


In [24]:
import bw2io as bi
import bw2data as bd
import bw2calc as bc

## Este notebook

El objetivo de este notebook es demostrar cómo trabajar con parámetros y Monte Carlo en Activity-Browser.

- Autor (y contacto):
    - Alvaro Hahn Menacho (alvaro.hahn-menacho@psi.ch)


## Ejercicio

Eres miembro de un equipo de expertos en análisis de ciclo de vida (ACV) que está revisando un modelo existente para mejorar su robustez y aplicabilidad en la toma de decisiones. El caso práctico se centra en comparar los impactos ambientales de un vehículo eléctrico (EV) y un vehículo diésel (ICEV), con énfasis en el ***Global Warming Potential (GWP)*** y los ***efectos de la formación de partículas en la salud humana***.

El ejercicio parte de un inventario existente. Sin embargo, este LCI se ha considerado inadecuado para la toma de decisiones debido a sus supuestos deterministas. El objetivo de este ejercicio es abordar estas preocupaciones creando un modelo de ACV más flexible que permita realizar análisis de sensibilidad. Durante una reunión de equipo, sus compañeros plantearon los siguientes puntos:

1. **Consumo energético de los vehículos eléctricos**: Un compañero señaló que el consumo establecido de unos 20 kWh por cada 100 km para el EV podría no ser exacto en todos los casos.
2. **Datos específicos de vehículos diésel**: Otra compañera señaló que el consumo de combustible de los vehículos diésel puede variar significativamente, y que el ACV debería tener en cuenta las diferencias entre los ICEV de alto y bajo consumo.
3. **Condiciones de conducción urbanas frente a no urbanas**: También destacaron que los impactos en la salud humana de las emisiones de los vehículos diésel varían significativamente entre entornos urbanos y no urbanos. Se quiere comprender cómo estos diferentes entornos de conducción influyen en los resultados de la evaluación.
4. **Correlación de las emisiones con el consumo de gasóleo**: Hay un interés particular en el seguimiento de las emisiones específicas vinculadas al consumo de diesel, especialmente $\text{CO}_{2}$, $\text{N}_{2}{O}$ y $\text{PM}_{2,5}$. El equipo quiere entender cómo influyen el punto 2 sobre estas emisiones en la evaluación de GWP.

| Parámetro       |Unidad| Distribución      | Loc   | Scale | Shape | Minimum | Maximum |
|-----------------|------|-------------------|-------|-------|-------|---------|---------|
|electricity_consumption|kWh/km|Normal|0.15|0.03||         |         |
|diesel_consumption|kg/km|Normal|0.06|0.02|       |         |         |
|urban_driving|%|Uniforme|       |       |       |0|1|
|co2_diesel|kg_co2/kg_diesel|Weibull|3|3.16|2|         |         |
|n2o_diesel|kg_n2o/kg_diesel|Lognormal|-7|0.05|       |         |         |
|pm25_diesel|kg_pm25/kg_diesel|Triangular|0.0015|       |       |0.0005|0.003|


Documentación sobre parámetros de incertidumbre y clases para muestreo Monte Carlo en BW2: https://stats-arrays.readthedocs.io/en/latest/

## 1. Establecer proyecto

- Esta actividad requiere un proyecto con una base de datos ecoinvent existente y su correspondiente biosfera.
- Como trabajaremos con Activity-Browser, necesitamos usar BW2 (no BW2.5)

In [25]:
bd.projects.set_current("mobility") #Creating/accessing your project.

In [26]:
bd.databases

Databases dictionary with 4 object(s):
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	parametric_LCA
	parametric_LCA_v1

## 2. Importar inventario Excel

In [27]:
imp = bi.ExcelImporter("parametric_lca_v0.xlsx")
imp.apply_strategies()
imp.match_database('ecoinvent-3.10-cutoff', fields=('name', 'unit', 'reference product', 'location'))
imp.statistics()

Extracted 1 worksheets in 0.04 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 5.64 seconds
Applying strategy: link_iterable_by_fields
6 datasets
20 exchanges
0 unlinked exchanges
  


(6, 20, 0)

In [28]:
imp.write_excel()

Wrote matching file to:
/home/gustavo/.local/share/Brightway3/mobility.7393daff1b0755731b3c165b370c333a/output/db-matching-parametric_LCA.xlsx


In [29]:
imp.write_database()

Writing activities to SQLite3 database:
0% [######] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 02/04/2025 21:38:35
  Finished: 02/04/2025 21:38:35
  Total time elapsed: 00:00:00
  CPU %: 0.00
  Memory %: 2.31
Created database: parametric_LCA


## 3. Importar inventario parametrizado

Después de construir tu nuevo inventario en 'parametric_lca_v1.xlsx'...

👓 La actividad "driving the EV" muestra cómo añadir parámetros y fórmulas  

In [30]:
imp = bi.ExcelImporter("parametric_lca_v1.xlsx")
imp.apply_strategies()
imp.match_database('ecoinvent-3.10-cutoff', fields=('name', 'unit', 'reference product', 'location'))
imp.statistics()

Extracted 1 worksheets in 0.02 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 5.96 seconds
Applying strategy: link_iterable_by_fields
6 datasets
22 exchanges
0 unlinked exchanges
  


(6, 22, 0)

---

#### ⚠️ **Unlinked exchanges?**

In [31]:
imp.write_excel()

Wrote matching file to:
/home/gustavo/.local/share/Brightway3/mobility.7393daff1b0755731b3c165b370c333a/output/db-matching-parametric_LCA_v1.xlsx


Esto suele ser de gran ayuda para detectar lo que ha ido mal.

---

#### 🚨 La sintaxis cambia un poco cuando queremos trabajar con parámetros

In [32]:
imp.write_project_parameters()

In [33]:
imp.write_database(activate_parameters=True)

Writing activities to SQLite3 database:
0% [######] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 02/04/2025 21:38:56
  Finished: 02/04/2025 21:38:56
  Total time elapsed: 00:00:00
  CPU %: 375.70
  Memory %: 2.40
Created database: parametric_LCA_v1


----

### **¡Es hora de continuar el análisis en AB!**